In [1]:
!rm -rf KernelOracle
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/mmarcotullio/KernelOracle.git
%cd KernelOracle
!git checkout -f main
!ls

Cloning into 'KernelOracle'...
remote: Enumerating objects: 1300, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 1300 (delta 101), reused 125 (delta 32), pack-reused 1064 (from 1)
Receiving objects: 100% (1300/1300), 62.04 MiB | 14.69 MiB/s, done.
Resolving deltas: 100% (411/411), done.
/content/KernelOracle
Already on 'main'
Your branch is up to date with 'origin/main'.
 data				      README.md
 LICENSE			      report.pdf
 lstm				      requirements.txt
 lstm_implementation_and_eval.ipynb   tcn
 lstm_training_and_evaluation.ipynb  'TCN_Training_&_Evaluation.ipynb'
 original_code


In [2]:
!pip -q install torch pandas numpy scikit-learn matplotlib tqdm

In [3]:
from tcn.models.tcn import TCNConfig, TCNNextPid
from tcn.utils.data import TraceWindowDataset, Vocab, batch_to_device
from tcn.utils.metrics import top1_accuracy, measure_inference_latency_ms

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

DATA_DIR = "/content/drive/MyDrive/trace_data"
print("Files in DATA_DIR:", os.listdir(DATA_DIR))

Files in DATA_DIR: ['dataset_meta.json', 'test_seen.csv', 'train.csv', 'test_unseen.csv', 'all.csv', 'cpu_bound_4p.csv', 'cpu_bound_8p.csv', 'io_mixed.csv', 'sysbench_cpu_8t.csv', 'sysbench_memory_4t.csv', 'hackbench_socket_large.csv', 'hackbench_pipe_large.csv', 'data_tcn']


In [6]:
# link for trace_data
!ln -s "/content/drive/MyDrive/trace_data" trace_data
!ls -lh trace_data
!head -n 2 trace_data/train.csv

lrwxrwxrwx 1 root root 33 Mar 18 02:32 trace_data -> /content/drive/MyDrive/trace_data
timestamp,time_diff,task_name,pid,cpu,prev_state,prev_comm,workload_type,trace_id
3710767.341761,0.000000000,code,861190,003,R,swapper/3,cpu_bound_4p,cpu_bound_4p_run1


In [7]:
import numpy as np
import json
import os

# Vocabulary construction
def build_vocab(df, vocab_out_path):
    df["pid_str"] = df["pid"].astype(str)
    unique_pids = sorted(df["pid_str"].unique())
    pid_to_idx = {"<UNK>": 0}
    for i, pid in enumerate(unique_pids, start=1):
        pid_to_idx[pid] = i
    unique_states = sorted(df["prev_state"].astype(str).unique())
    state_to_idx = {state: i for i, state in enumerate(unique_states)}

    vocab = {
        "pid_to_idx": pid_to_idx,
        "state_to_idx": state_to_idx,
    }

    with open(vocab_out_path, "w") as f:
        json.dump(vocab, f)

    return vocab



def load_vocab(vocab_path):
    with open(vocab_path, "r") as f:
        return json.load(f)


def encode_df(df, vocab):
    df["pid_str"] = df["pid"].astype(str)
    pid_to_idx = vocab["pid_to_idx"]
    if "<UNK>" not in pid_to_idx:
        pid_to_idx["<UNK>"] = 0
    unk_idx = pid_to_idx["<UNK>"]
    df["pid_idx"] = (
        df["pid_str"]
        .map(lambda x: pid_to_idx.get(x, unk_idx))
        .astype(np.int64)
    )
    state_to_idx = vocab["state_to_idx"]
    df["state_idx"] = (
        df["prev_state"].astype(str)
        .map(lambda x: state_to_idx.get(x, 0))
        .astype(np.int64)
    )

    return df

In [8]:
rm -rf tcn/artifacts/*

In [9]:
%%bash

FILE="tcn/scripts/preprocess.py"


sed -i 's/df\["pid_idx"\] = df\["pid_str"\].map(vocab\["pid_to_idx"\]).astype(np.int64)/pid_to_idx = vocab["pid_to_idx"]\n    if "<UNK>" not in pid_to_idx:\n        pid_to_idx["<UNK>"] = 0\n    unk_idx = pid_to_idx["<UNK>"]\n    df["pid_idx"] = df["pid_str"].map(lambda x: pid_to_idx.get(x, unk_idx)).astype(np.int64)/' $FILE

echo "Patched preprocess.py"

Patched preprocess.py


In [10]:
!sed -n '35,65p' tcn/scripts/preprocess.py

        "pid_to_idx": pid_to_idx,
        "idx_to_pid": idx_to_pid,
        "state_to_idx": state_to_idx,
        "idx_to_state": idx_to_state,
    }


def encode_df(df: pd.DataFrame, vocab: Dict) -> pd.DataFrame:
    df = df.copy()

    df["pid_str"] = df["pid"].astype(str)
    pid_to_idx = vocab["pid_to_idx"]
    if "<UNK>" not in pid_to_idx:
        pid_to_idx["<UNK>"] = 0
    unk_idx = pid_to_idx["<UNK>"]
    df["pid_idx"] = df["pid_str"].map(lambda x: pid_to_idx.get(x, unk_idx)).astype(np.int64)

    df["prev_state_str"] = df["prev_state"].astype(str).fillna("UNK")

    if "UNK" not in vocab["state_to_idx"]:

        vocab["state_to_idx"]["UNK"] = len(vocab["state_to_idx"])
        vocab["idx_to_state"][vocab["state_to_idx"]["UNK"]] = "UNK"

    df["state_idx"] = df["prev_state_str"].map(lambda s: vocab["state_to_idx"].get(s, vocab["state_to_idx"]["UNK"])).astype(np.int64)


    df["cpu"] = pd.to_numeric(df.get("cpu", 0), errors="coerce").fillna(0.0).astype(np.float32)
    df["tim

In [11]:
!rm -rf tcn/artifacts/*
!ls -lh tcn/artifacts

total 0


In [12]:
#Preprocessing train csv
!python tcn/scripts/preprocess.py \
 --csv trace_data/train.csv \
 --out tcn/artifacts/train.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4

Saved tcn/artifacts/train.npz
Windows: 1812164, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [13]:
# Preprocessing test seen csv
!python tcn/scripts/preprocess.py \
 --csv trace_data/test_seen.csv \
 --out tcn/artifacts/test_seen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4 \
 --use_existing_vocab

Saved tcn/artifacts/test_seen.npz
Windows: 792463, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [14]:
# Preprocessing test unseen csv
!python tcn/scripts/preprocess.py \
 --csv trace_data/test_unseen.csv \
 --out tcn/artifacts/test_unseen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4 \
 --use_existing_vocab

Saved tcn/artifacts/test_unseen.npz
Windows: 2172560, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [15]:
!ls -lrt trace_data

lrwxrwxrwx 1 root root 33 Mar 18 02:32 trace_data -> /content/drive/MyDrive/trace_data


In [16]:
# Preprocessing workloadad
workloads = ["cpu_bound_4p", "cpu_bound_8p", "hackbench_pipe_large",
             "hackbench_socket_large", "io_mixed", "sysbench_cpu_8t",
             "sysbench_memory_4t"]

for w in workloads:
    csv_path = f"trace_data/{w}.csv"
    out_path = f"tcn/artifacts/{w}.npz"

    cmd = f"""
    python tcn/scripts/preprocess.py \
        --csv {csv_path} \
        --out {out_path} \
        --vocab_out tcn/artifacts/vocab.json \
        --seq_len 64 \
        --stride 4 \
        --use_existing_vocab
    """

    print(f"\nProcessing workload: {w}")
    os.system(cmd)



Processing workload: cpu_bound_4p

Processing workload: cpu_bound_8p

Processing workload: hackbench_pipe_large

Processing workload: hackbench_socket_large

Processing workload: io_mixed

Processing workload: sysbench_cpu_8t

Processing workload: sysbench_memory_4t


In [17]:
!ls -lh tcn/artifacts

total 119M
-rw-r--r-- 1 root root 538K Mar 18 02:35 cpu_bound_4p.npz
-rw-r--r-- 1 root root 264K Mar 18 02:35 cpu_bound_8p.npz
-rw-r--r-- 1 root root 9.0M Mar 18 02:35 hackbench_pipe_large.npz
-rw-r--r-- 1 root root 4.3M Mar 18 02:36 hackbench_socket_large.npz
-rw-r--r-- 1 root root 329K Mar 18 02:36 io_mixed.npz
-rw-r--r-- 1 root root 547K Mar 18 02:36 sysbench_cpu_8t.npz
-rw-r--r-- 1 root root 456K Mar 18 02:36 sysbench_memory_4t.npz
-rw-r--r-- 1 root root  16M Mar 18 02:34 test_seen.npz
-rw-r--r-- 1 root root  38M Mar 18 02:35 test_unseen.npz
-rw-r--r-- 1 root root  51M Mar 18 02:33 train.npz
-rw-r--r-- 1 root root 124K Mar 18 02:33 vocab.json


In [18]:
# Training the model
!python -m tcn.train_tcn \
  --train_npz tcn/artifacts/train.npz \
  --test_seen_npz tcn/artifacts/test_seen.npz \
  --test_unseen_npz tcn/artifacts/test_unseen.npz \
  --vocab tcn/artifacts/vocab.json \
  --epochs 5 \
  --batch_size 128 \
  --lr 0.001 \
  --save_dir tcn/checkpoints

epoch=1 loss=4.6540 time=172.2s | seen_acc=0.9161 | unseen_acc=0.9843
Saved best checkpoint -> tcn/checkpoints/tcn_nextpid_best.pt
epoch=2 loss=4.2481 time=172.2s | seen_acc=0.9180 | unseen_acc=0.9842
Saved best checkpoint -> tcn/checkpoints/tcn_nextpid_best.pt
epoch=3 loss=4.1435 time=169.8s | seen_acc=0.9181 | unseen_acc=0.9832
Saved best checkpoint -> tcn/checkpoints/tcn_nextpid_best.pt
epoch=4 loss=4.0830 time=173.3s | seen_acc=0.9165 | unseen_acc=0.9839
epoch=5 loss=4.0412 time=172.5s | seen_acc=0.9172 | unseen_acc=0.9826
cpu_bound_4p              acc=0.4611 samples=16658
cpu_bound_8p              acc=0.4214 samples=8961
hackbench_pipe_large      acc=0.9693 samples=504252
hackbench_socket_large    acc=0.9470 samples=214731
io_mixed                  acc=0.5194 samples=12369
sysbench_cpu_8t           acc=0.4888 samples=19636
sysbench_memory_4t        acc=0.4468 samples=15762


In [20]:
import numpy as np

data = np.load("tcn/artifacts/train.npz")
print(data.files)

['pid', 'cont', 'state', 'y', 'seq_len']


In [21]:
import torch
import time
from tcn.models.tcn import TCNConfig, TCNNextPid
from tcn.utils.data import TraceWindowDataset, Vocab, batch_to_device
from torch.utils.data import DataLoader

# Force to run evaluation on cpu
device = torch.device("cpu")
torch.set_num_threads(1)

print("Using device:", device)
vocab = Vocab.load("tcn/artifacts/vocab.json")

ckpt = torch.load(
    "tcn/checkpoints/tcn_nextpid_best.pt",
    map_location=device
)

cfg = TCNConfig(**ckpt["cfg"])
model = TCNNextPid(cfg).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

ds = TraceWindowDataset("tcn/artifacts/train.npz")
loader = DataLoader(ds, batch_size=128, shuffle=False)

batch = next(iter(loader))
batch = batch_to_device(batch, device)

pid = batch["pid"]
cont = batch["cont"]
state = batch["state"]

# sanity check for the right device type
print("Model device:", next(model.parameters()).device)
print("pid device:", pid.device)
print("cont device:", cont.device)
print("state device:", state.device)

print("pid shape:", pid.shape)
print("cont shape:", cont.shape)
print("state shape:", state.shape)

Using device: cpu
Model device: cpu
pid device: cpu
cont device: cpu
state device: cpu
pid shape: torch.Size([128, 64])
cont shape: torch.Size([128, 64, 2])
state shape: torch.Size([128, 64])


In [22]:
# Computing forward latency, inference latency, and throughput
warmup = 20
iters = 100

for _ in range(warmup):
    _ = model(pid, cont, state)

start = time.perf_counter()

for _ in range(iters):
    _ = model(pid, cont, state)

end = time.perf_counter()

lat_ms = (end - start) / iters * 1000


batch_size = pid.shape[0]

print(f"\nAvg forward latency per batch: {lat_ms:.3f} ms (CPU)")
print(f"Latency per sample: {(lat_ms / batch_size):.6f} ms")

throughput = batch_size / (lat_ms / 1000)
print(f"Throughput: {throughput:.2f} samples/sec")


Avg forward latency per batch: 175.409 ms (CPU)
Latency per sample: 1.370386 ms
Throughput: 729.72 samples/sec
